# Index-Anchored Population Detection Results

This notebook reviews the completed population detection simulations and performs lightweight behavioral matching. Running all cells reads the six authoritative CSV outputs under `data/detection/`, checks their coverage, creates summary visualizations, and fits constrained operating-state mixtures. It does not import NEURON or launch threshold searches or population simulations.

The true-regime response table uses each repetition's own index-axon amplitudes. The CV0 response table replays the frequency-specific CV0-derived amplitudes across every CV and population axon.

In [ ]:
# ---------------- Imports and paths ----------------

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
REPO_ROOT = next((path for path in (cwd, *cwd.parents) if (path / 'data').is_dir() and (path / 'notebooks').is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError('Run this notebook from inside the scs-periodicity repository.')

OUTPUT_DIR = REPO_ROOT / 'data' / 'detection'
INDEX_THRESHOLDS_CSV = OUTPUT_DIR / 'index_axon_thresholds.csv'
INDEX_AXON_CSV = OUTPUT_DIR / 'index_axon.csv'
INDEX_REGIMES_CSV = OUTPUT_DIR / 'index_regime_amplitudes.csv'
INDEX_CV0_THRESH_CSV = OUTPUT_DIR / 'index_cv0_threshold_max.csv'
POPULATION_RESPONSES_CSV = OUTPUT_DIR / 'population_responses.csv'
POPULATION_CV0_RESPONSES_CSV = OUTPUT_DIR / 'population_cv0_based_responses.csv'

OUTPUT_PATHS = (
    INDEX_THRESHOLDS_CSV, INDEX_AXON_CSV, INDEX_REGIMES_CSV,
    INDEX_CV0_THRESH_CSV, POPULATION_RESPONSES_CSV,
    POPULATION_CV0_RESPONSES_CSV,
)
missing = [path for path in OUTPUT_PATHS if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing completed population outputs:\n' + '\n'.join(map(str, missing)))

print(f'Population detection outputs: {OUTPUT_DIR}')

## 1. Index axon and amplitude states

The four smaller tables fit comfortably in memory. A missing reference threshold means that an axon did not spike within the standard reference search range; it is not a missing population-response simulation.

In [ ]:
index_thresholds = pd.read_csv(INDEX_THRESHOLDS_CSV)
index_axon = pd.read_csv(INDEX_AXON_CSV)
index_regimes = pd.read_csv(INDEX_REGIMES_CSV)
index_cv0 = pd.read_csv(INDEX_CV0_THRESH_CSV)
cv0_count_columns = [column for column in index_cv0 if column.endswith('_n_index_spikes')]
if len(cv0_count_columns) != 9 or index_cv0[cv0_count_columns].isna().any().any():
    raise RuntimeError('CV0 threshold metadata lacks complete realized index-spike counts; regenerate index_cv0_threshold_max.csv')

index_summary = pd.DataFrame({
    'quantity': [
        'population axons', 'finite reference thresholds',
        'no spike in reference search', 'index-regime rows',
        'CV values', 'frequencies', 'repetitions', 'regimes',
    ],
    'value': [
        index_thresholds['axon_id'].nunique(),
        index_thresholds['I_first_uA'].notna().sum(),
        index_thresholds['I_first_uA'].isna().sum(),
        len(index_regimes), index_regimes['cv'].nunique(),
        index_regimes['freq'].nunique(), index_regimes['rep_index'].nunique(),
        index_regimes['thresh_type'].nunique(),
    ],
})
display(index_axon)
display(index_summary)
display(index_cv0)
display(Markdown('### Realized CV0 index-spike counts at each retained amplitude'))
display(index_cv0[['freq', *cv0_count_columns]])

## 2. Reference thresholds and index-regime amplitudes

In [ ]:
# ---------------- Anatomically registered plot helpers ----------------

DEPTH_ORDER = ['S', 'M', 'D']
DEPTH_LABELS = {'S': 'Superficial', 'M': 'Middle', 'D': 'Deep'}
COLUMN_ORDER = list(range(1, 14))  # 01 is most lateral; 13 is most medial.
DIAMETER_ORDER = sorted(index_thresholds['diameter_um'].unique(), reverse=True)
PHASE_ORDER = (
    index_thresholds[['phase_id', 'phase_fraction']]
    .drop_duplicates()
    .sort_values('phase_fraction')['phase_id']
    .tolist()
)
PHASE_FRACTIONS = (
    index_thresholds[['phase_id', 'phase_fraction']]
    .drop_duplicates()
    .set_index('phase_id')['phase_fraction']
    .to_dict()
)
ANATOMICAL_ROWS = pd.MultiIndex.from_product(
    [DEPTH_ORDER, DIAMETER_ORDER], names=['depth', 'diameter_um']
)
ANATOMICAL_PHASE_ROWS = pd.MultiIndex.from_product(
    [DEPTH_ORDER, DIAMETER_ORDER, PHASE_ORDER],
    names=['depth', 'diameter_um', 'phase_id'],
)


def add_anatomical_coordinates(table, cutline_col, diameter_col, phase_col):
    """Add depth/column/phase coordinates without changing source data."""
    annotated = table.copy()
    cutline = annotated[cutline_col].astype(str)
    annotated['_depth'] = cutline.str[0]
    annotated['_column'] = cutline.str[1:].astype(int)
    annotated['_diameter_um'] = annotated[diameter_col].astype(float)
    annotated['_phase_id'] = annotated[phase_col].astype(str)
    return annotated


def anatomical_phase_matrix(table, value_col, *, cutline_col, diameter_col, phase_col):
    """Return one matrix with ph0/ph1 stacked within each anatomical cell."""
    annotated = add_anatomical_coordinates(table, cutline_col, diameter_col, phase_col)
    matrix = annotated.pivot(
        index=['_depth', '_diameter_um', '_phase_id'],
        columns='_column',
        values=value_col,
    )
    matrix.index = matrix.index.set_names(['depth', 'diameter_um', 'phase_id'])
    return matrix.reindex(index=ANATOMICAL_PHASE_ROWS, columns=COLUMN_ORDER)


def plot_anatomical_phase_stack(
    table,
    value_col,
    *,
    cutline_col,
    diameter_col,
    phase_col,
    title,
    colorbar_label,
    cmap='viridis_r',
    vmin=None,
    vmax=None,
    formatter=lambda value: f'{value:.0f}',
    annotate=True,
    highlight=None,
):
    """Plot ph0 above ph1 as two half-height rectangles per anatomy cell."""
    matrix = anatomical_phase_matrix(
        table,
        value_col,
        cutline_col=cutline_col,
        diameter_col=diameter_col,
        phase_col=phase_col,
    )
    values = matrix.to_numpy(dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        raise ValueError(f'No finite {value_col} values are available for this view.')
    vmin = float(finite.min()) if vmin is None else float(vmin)
    vmax = float(finite.max()) if vmax is None else float(vmax)
    color_map = plt.get_cmap(cmap).copy()
    color_map.set_bad('#d9d9d9')

    fig, axis = plt.subplots(figsize=(14, 12), constrained_layout=True)
    image = axis.imshow(
        np.ma.masked_invalid(values),
        aspect='auto',
        cmap=color_map,
        vmin=vmin,
        vmax=vmax,
        interpolation='none',
    )
    axis.set_xticks(
        range(len(COLUMN_ORDER)), [f'{column:02d}' for column in COLUMN_ORDER]
    )
    axis.set_xlabel('Mediolateral column (01 lateral → 13 medial)')
    axis.set_yticks(
        range(len(ANATOMICAL_PHASE_ROWS)),
        [
            f'{DEPTH_LABELS[depth]} · {diameter:g} µm · {phase_id}'
            for depth, diameter, phase_id in ANATOMICAL_PHASE_ROWS
        ],
        fontsize=7,
    )
    axis.set_ylabel('Anatomical depth, MRG diameter, and nodal phase')
    phase_note = ' · '.join(
        f'{phase_id} = {PHASE_FRACTIONS[phase_id]:.2f} internode'
        for phase_id in PHASE_ORDER
    )
    axis.set_title(phase_note, fontsize=10, pad=8)

    axis.set_xticks(np.arange(-0.5, len(COLUMN_ORDER), 1), minor=True)
    axis.set_yticks(np.arange(-0.5, len(ANATOMICAL_PHASE_ROWS), 1), minor=True)
    axis.grid(which='minor', color='white', linewidth=0.3, alpha=0.45)
    axis.tick_params(which='minor', bottom=False, left=False)
    # A phase-pair occupies the height of one former square. Thin lines divide
    # ph0 from ph1; stronger lines divide diameters and anatomical depths.
    for diameter_boundary in range(1, len(DEPTH_ORDER) * len(DIAMETER_ORDER)):
        axis.axhline(diameter_boundary * len(PHASE_ORDER) - 0.5, color='#555555', linewidth=0.7)
    rows_per_depth = len(DIAMETER_ORDER) * len(PHASE_ORDER)
    for depth_boundary in range(1, len(DEPTH_ORDER)):
        axis.axhline(depth_boundary * rows_per_depth - 0.5, color='black', linewidth=2.2)

    if annotate:
        span = vmax - vmin
        for row in range(values.shape[0]):
            for column in range(values.shape[1]):
                value = values[row, column]
                if np.isfinite(value):
                    normalized = 0.5 if span == 0 else (value - vmin) / span
                    red, green, blue, _ = color_map(np.clip(normalized, 0, 1))
                    luminance = 0.2126 * red + 0.7152 * green + 0.0722 * blue
                    text_color = 'black' if luminance > 0.55 else 'white'
                    axis.text(
                        column,
                        row,
                        formatter(value),
                        ha='center',
                        va='center',
                        fontsize=5.5,
                        color=text_color,
                    )

    if highlight:
        depth = str(highlight['cutline_id'])[0]
        column = int(str(highlight['cutline_id'])[1:])
        row = ANATOMICAL_PHASE_ROWS.get_loc(
            (depth, float(highlight['diameter_um']), str(highlight['phase_id']))
        )
        axis.add_patch(Rectangle(
            (COLUMN_ORDER.index(column) - 0.48, row - 0.48),
            0.96,
            0.96,
            fill=False,
            edgecolor='red',
            linewidth=2.5,
        ))

    fig.suptitle(title, fontsize=15)
    fig.colorbar(image, ax=axis, label=colorbar_label, shrink=0.88, pad=0.02)
    return fig, axis, matrix


index_row = index_axon.iloc[0]
index_highlight = {
    'cutline_id': index_row['cutline_id'],
    'diameter_um': index_row['diameter_um'],
    'phase_id': index_row['phase_id'],
}
threshold_fig, threshold_axis, threshold_map = plot_anatomical_phase_stack(
    index_thresholds,
    'I_first_uA',
    cutline_col='cutline_id',
    diameter_col='diameter_um',
    phase_col='phase_id',
    title='Standard-reference first-spike threshold',
    colorbar_label='Current amplitude (µA)',
    cmap='viridis_r',
    formatter=lambda value: f'{value:.0f}',
    highlight=index_highlight,
)
plt.show()


REGIME_ORDER = ['I_first', 'I_volley5', 'I_volley10', 'I_5', 'I_10', 'I_15', 'I_25', 'I_50', 'I_all']
VIEW_FREQ_HZ = 200
amplitude_summary = (
    index_regimes[index_regimes['freq'].eq(VIEW_FREQ_HZ)]
    .groupby(['cv', 'thresh_type'], observed=True)['amp_uA']
    .agg(median='median', q05=lambda values: values.quantile(0.05), q95=lambda values: values.quantile(0.95))
    .reset_index()
)
fig, ax = plt.subplots(figsize=(9, 5))
for regime in REGIME_ORDER:
    values = amplitude_summary[amplitude_summary['thresh_type'].eq(regime)].sort_values('cv')
    ax.plot(values['cv'], values['median'], marker='o', markersize=3, label=regime)
ax.set(xlabel='Coefficient of variation', ylabel='Index-axon amplitude (µA)', title=f'Index-regime median amplitudes at {VIEW_FREQ_HZ} Hz')
ax.grid(alpha=0.25)
ax.legend(ncol=3, fontsize=8)
fig.tight_layout()
plt.show()

### Nodal-phase threshold separation

The paired phase diagnostic quantifies how much the standard-reference threshold changes when the nodes are shifted by half an internodal spacing on the same cutline and diameter. It is retained because nodal phase can materially affect which nominally colocated axon becomes the index axon. Tables summarize matched pairs by diameter, depth, and mediolateral column; the full cutline-by-diameter map retains spatial detail. Positive signed gaps mean that ph1 has the lower threshold.

In [ ]:
PHASE_THRESHOLD_PAIRS_CSV = OUTPUT_DIR / 'reference_phase_threshold_pairs.csv'
PHASE_THRESHOLD_SUMMARY_CSV = OUTPUT_DIR / 'reference_phase_threshold_summary.csv'
phase_threshold_pairs = (
    index_thresholds.pivot(
        index=['cutline_id', 'diameter_um'], columns='phase_id', values='I_first_uA'
    )
    .dropna(subset=['ph0', 'ph1'])
    .reset_index()
)
phase_threshold_pairs['ph0_minus_ph1_uA'] = phase_threshold_pairs['ph0'] - phase_threshold_pairs['ph1']
phase_threshold_pairs['absolute_gap_uA'] = phase_threshold_pairs['ph0_minus_ph1_uA'].abs()
phase_threshold_pairs['gap_percent_of_lower_threshold'] = (
    100 * phase_threshold_pairs['absolute_gap_uA']
    / phase_threshold_pairs[['ph0', 'ph1']].min(axis=1)
)
phase_threshold_pairs['signed_gap_percent_of_lower_threshold'] = (
    100 * phase_threshold_pairs['ph0_minus_ph1_uA']
    / phase_threshold_pairs[['ph0', 'ph1']].min(axis=1)
)
phase_threshold_pairs['depth_row'] = phase_threshold_pairs['cutline_id'].str[0]
phase_threshold_pairs['mediolateral_column'] = phase_threshold_pairs['cutline_id'].str[1:].astype(int)
phase_threshold_pairs.to_csv(PHASE_THRESHOLD_PAIRS_CSV, index=False)

def summarize_phase_gap(frame, group_column, stratum_type):
    summary = (
        frame.groupby(group_column, observed=True, sort=True)
        .agg(
            matched_pairs=('cutline_id', 'size'),
            ph1_lower_count=('ph0_minus_ph1_uA', lambda values: int(values.gt(0).sum())),
            median_gap_uA=('absolute_gap_uA', 'median'),
            median_gap_percent=('gap_percent_of_lower_threshold', 'median'),
            q25_gap_percent=('gap_percent_of_lower_threshold', lambda values: values.quantile(0.25)),
            q75_gap_percent=('gap_percent_of_lower_threshold', lambda values: values.quantile(0.75)),
            min_gap_percent=('gap_percent_of_lower_threshold', 'min'),
            max_gap_percent=('gap_percent_of_lower_threshold', 'max'),
        )
        .reset_index()
        .rename(columns={group_column: 'stratum'})
    )
    summary.insert(0, 'stratum_type', stratum_type)
    return summary

phase_threshold_summary = pd.concat([
    summarize_phase_gap(phase_threshold_pairs, 'diameter_um', 'diameter_um'),
    summarize_phase_gap(phase_threshold_pairs, 'depth_row', 'depth_row'),
    summarize_phase_gap(phase_threshold_pairs, 'mediolateral_column', 'mediolateral_column'),
], ignore_index=True)
phase_threshold_summary.to_csv(PHASE_THRESHOLD_SUMMARY_CSV, index=False)
phase_gap_summary = pd.DataFrame({
    'paired cutline/diameter conditions': [len(phase_threshold_pairs)],
    'ph1 lower threshold': [int(phase_threshold_pairs['ph0_minus_ph1_uA'].gt(0).sum())],
    'ph0 lower threshold': [int(phase_threshold_pairs['ph0_minus_ph1_uA'].lt(0).sum())],
    'median absolute gap (uA)': [phase_threshold_pairs['absolute_gap_uA'].median()],
    'median gap (% of lower)': [phase_threshold_pairs['gap_percent_of_lower_threshold'].median()],
})
index_phase_pair = phase_threshold_pairs[
    phase_threshold_pairs['cutline_id'].eq(index_axon.loc[0, 'cutline_id'])
    & phase_threshold_pairs['diameter_um'].eq(index_axon.loc[0, 'diameter_um'])
]
display(phase_gap_summary)
display(phase_threshold_summary[phase_threshold_summary['stratum_type'].eq('diameter_um')])
display(phase_threshold_summary[phase_threshold_summary['stratum_type'].eq('depth_row')])
display(phase_threshold_summary[phase_threshold_summary['stratum_type'].eq('mediolateral_column')])
display(index_phase_pair)

anatomical_cutline_order = (
    [f'S{column:02d}' for column in range(1, 14)]
    + [f'M{column:02d}' for column in range(1, 14)]
    + [f'D{column:02d}' for column in range(5, 14)]
)
phase_gap_matrix = (
    phase_threshold_pairs.pivot(
        index='cutline_id', columns='diameter_um',
        values='signed_gap_percent_of_lower_threshold',
    )
    .reindex(index=anatomical_cutline_order, columns=DIAMETER_ORDER)
)
fig, ax = plt.subplots(figsize=(9, 11))
image = ax.imshow(phase_gap_matrix.to_numpy(), aspect='auto', cmap='viridis')
ax.set(
    xticks=np.arange(len(phase_gap_matrix.columns)),
    yticks=np.arange(len(phase_gap_matrix.index)),
    xticklabels=[f'{diameter:g}' for diameter in phase_gap_matrix.columns],
    yticklabels=phase_gap_matrix.index,
    xlabel='MRG fiber diameter (um)',
    ylabel='COMSOL cutline (lateral to medial within depth row)',
    title='Matched nodal-phase threshold gap: 100 x (ph0 - ph1) / lower threshold',
)
for row_index, cutline_id in enumerate(phase_gap_matrix.index):
    for column_index, diameter_um in enumerate(phase_gap_matrix.columns):
        value = phase_gap_matrix.loc[cutline_id, diameter_um]
        if np.isfinite(value):
            ax.text(column_index, row_index, f'{value:.1f}', ha='center', va='center', fontsize=7,
                    color='white' if value > 9 else 'black')
index_row = phase_gap_matrix.index.get_loc(index_axon.loc[0, 'cutline_id'])
index_column = phase_gap_matrix.columns.get_loc(index_axon.loc[0, 'diameter_um'])
ax.add_patch(Rectangle((index_column - 0.5, index_row - 0.5), 1, 1, fill=False, edgecolor='red', linewidth=2.5))
colorbar = fig.colorbar(image, ax=ax, pad=0.02)
colorbar.set_label('Signed phase-threshold gap (%)')
fig.tight_layout()
plt.show()

print('Wrote:')
print(f'  {PHASE_THRESHOLD_PAIRS_CSV}')
print(f'  {PHASE_THRESHOLD_SUMMARY_CSV}')

## 2b. High-resolution CV0 index operating-state pilot

This diagnostic is intentionally separate from the production threshold table. It reruns only the index axon for the Slack repetition and one representative regular repetition at 20, 50, 100, and 200 Hz, using a 0.01 µA threshold tolerance and a dense local sweep around every old and refined transition.

The outputs distinguish four possibilities that should not be conflated: labels with mathematically identical spike-count targets, thresholds merged by the old 0.1 µA tolerance, multiple targets crossed by a genuine response jump, and nonmonotonic spike-count changes with increasing amplitude. The proposed amplitude table is CV0-only and does **not** apply the former all-CV `I_first`/`I_volley5` safeguard.

Silence is retained as a response state. A zero population spike/event count may still carry rate evidence when the endpoint templates are separable; an unavailable silence-LV timing cue contributes zero evidence and is marked unavailable explicitly.

**Selected production specification:** CV0 is one generated exact-periodic waveform copied across all 100 historical repetition labels, population thresholds use 0.01 µA tolerance, and `rep001_slack` at CV0 is a compatibility label rather than a second waveform. The Slack-versus-regular pilot remains a provenance diagnostic for why this choice was made.


In [ ]:
# ---------------- Index operating-state pilot diagnostics ----------------

PILOT_DIR = REPO_ROOT / 'results' / 'detection' / 'index_operating_state_pilot'
PILOT_SAMPLES_CSV = PILOT_DIR / 'index_operating_state_search_samples.csv'
PILOT_THRESHOLDS_CSV = PILOT_DIR / 'index_operating_state_threshold_comparison.csv'
PILOT_SUMMARY_CSV = PILOT_DIR / 'index_operating_state_condition_summary.csv'
PILOT_COLLISIONS_CSV = PILOT_DIR / 'index_operating_state_definition_collisions.csv'
PILOT_NONMONOTONIC_CSV = PILOT_DIR / 'index_operating_state_nonmonotonic_steps.csv'
PILOT_PROPOSAL_CSV = PILOT_DIR / 'index_cv0_threshold_proposal.csv'

pilot_paths = [
    PILOT_SAMPLES_CSV, PILOT_THRESHOLDS_CSV, PILOT_SUMMARY_CSV,
    PILOT_COLLISIONS_CSV, PILOT_NONMONOTONIC_CSV, PILOT_PROPOSAL_CSV,
]
pilot_missing = [path for path in pilot_paths if not path.is_file()]
if pilot_missing:
    print('Index operating-state pilot results are not present yet.')
    print('Submit on Longleaf with:')
    print('  bash slurm/detection/run_index_operating_state_pilot.sh')
    print('Then download:')
    print(f'  {PILOT_DIR.relative_to(REPO_ROOT)}')
else:
    pilot_samples = pd.read_csv(PILOT_SAMPLES_CSV)
    pilot_thresholds = pd.read_csv(PILOT_THRESHOLDS_CSV)
    pilot_summary = pd.read_csv(PILOT_SUMMARY_CSV)
    pilot_collisions = pd.read_csv(PILOT_COLLISIONS_CSV)
    pilot_nonmonotonic = pd.read_csv(PILOT_NONMONOTONIC_CSV)
    pilot_proposal = pd.read_csv(PILOT_PROPOSAL_CSV)

    print('Definitionally identical targets (not resolvable by search tolerance):')
    display(pilot_collisions)
    print('Condition-level numerical and nonmonotonicity diagnostics:')
    display(pilot_summary.sort_values(['freq', 'representative_rep_index']))
    print('Every adjacent 0.01 µA step with a decreasing spike count:')
    display(pilot_nonmonotonic if len(pilot_nonmonotonic) else pd.DataFrame({'result': ['No local decreases detected']}))

    regime_order = ['I_first', 'I_volley5', 'I_volley10', 'I_5', 'I_10', 'I_15', 'I_25', 'I_50', 'I_all']
    regime_colors = dict(zip(regime_order, plt.cm.tab10.colors[:len(regime_order)]))
    frequencies = [20, 50, 100, 200]
    representatives = [0, 1]
    fig, axes = plt.subplots(len(frequencies), len(representatives), figsize=(16, 14), squeeze=False)
    for row_index, frequency in enumerate(frequencies):
        for column_index, rep_index in enumerate(representatives):
            ax = axes[row_index, column_index]
            block = pilot_samples[
                np.isclose(pilot_samples['freq'], frequency)
                & (pilot_samples['representative_rep_index'] == rep_index)
                & pilot_samples['is_local_sweep'].astype(bool)
            ].sort_values('amp_uA')
            if block.empty:
                ax.set_visible(False)
                continue
            amplitudes = block['amp_uA'].to_numpy(float)
            counts = block['n_index_spikes'].to_numpy(float)
            step = float(block['sweep_step_uA'].iloc[0])
            breaks = np.flatnonzero(np.diff(amplitudes) > 1.05 * step) + 1
            for segment in np.split(np.arange(len(block)), breaks):
                ax.step(amplitudes[segment], counts[segment], where='post', color='black', lw=1.4)
                ax.scatter(amplitudes[segment], counts[segment], color='black', s=8, zorder=3)
            threshold_block = pilot_thresholds[
                np.isclose(pilot_thresholds['freq'], frequency)
                & (pilot_thresholds['representative_rep_index'] == rep_index)
            ]
            for regime in regime_order:
                selected = threshold_block[threshold_block['thresh_type'] == regime]
                if selected.empty:
                    continue
                item = selected.iloc[0]
                ax.axvline(item['refined_amp_uA'], color=regime_colors[regime], lw=1.0, alpha=.8)
                ax.scatter(item['refined_amp_uA'], item['refined_n_index_spikes'],
                           color=regime_colors[regime], s=34, zorder=4,
                           label=regime if row_index == 0 and column_index == 0 else None)
            rep_name = 'Slack rep 0' if rep_index == 0 else 'regular reps 1–99'
            drops = pilot_summary[
                np.isclose(pilot_summary['freq'], frequency)
                & (pilot_summary['representative_rep_index'] == rep_index)
            ]['n_nonmonotonic_steps'].iloc[0]
            ax.set_title(f'{frequency} Hz | {rep_name} | local decreases={int(drops)}')
            ax.set_xlabel('Amplitude (µA)')
            ax.set_ylabel('Index-axon spike count')
            ax.grid(alpha=.2)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='lower center', ncol=5, frameon=False)
    fig.suptitle('Fine CV0 index response staircases around operating-state transitions', y=.995)
    fig.tight_layout(rect=(0, .055, 1, .98))
    plt.show()

    comparison_columns = [
        'freq', 'representative_rep_index', 'thresh_type', 'target_spikes',
        'definition_equivalent_regimes', 'old_amp_uA', 'refined_amp_uA',
        'amp_shift_uA', 'refined_n_index_spikes',
        'target_exactly_observed_in_local_sweep', 'target_skipped_at_refined_crossing',
        'plateau_sampled_width_uA',
    ]
    print('Old versus refined threshold definitions:')
    display(
        pilot_thresholds[comparison_columns]
        .assign(thresh_type=lambda x: pd.Categorical(x['thresh_type'], regime_order, ordered=True))
        .sort_values(['freq', 'representative_rep_index', 'thresh_type'])
    )
    print('CV0-only amplitude proposal (diagnostic; not yet a production input):')
    display(pilot_proposal)


## 3. Chunked population-response summaries

Each response CSV is several gigabytes, so this cell never loads a full response table. It streams chunks and retains only one aggregated row per target axon, CV, frequency, and regime. `recruitment_probability` is the fraction of the 100 repetitions in which that axon produced at least one terminal-node spike.

In [ ]:
def summarize_population_csv(path, mode, chunksize=250_000):
    group_columns = [
        'target_axon_id', 'target_cutline_id', 'target_diameter_um',
        'cv', 'freq', 'thresh_type',
    ]
    use_columns = [*group_columns, 'amp_uA', 'n_spikes']
    partials = []
    total_rows = missing_amplitudes = missing_spike_counts = 0
    for chunk in pd.read_csv(path, usecols=use_columns, chunksize=chunksize):
        total_rows += len(chunk)
        missing_amplitudes += int(chunk['amp_uA'].isna().sum())
        missing_spike_counts += int(chunk['n_spikes'].isna().sum())
        chunk['recruited'] = chunk['n_spikes'].gt(0).astype(np.int8)
        partials.append(
            chunk.groupby(group_columns, observed=True, sort=False)
            .agg(
                n_trials=('n_spikes', 'size'),
                recruited_trials=('recruited', 'sum'),
                spike_sum=('n_spikes', 'sum'),
            )
            .reset_index()
        )
    summary = (
        pd.concat(partials, ignore_index=True)
        .groupby(group_columns, observed=True, sort=False)[['n_trials', 'recruited_trials', 'spike_sum']]
        .sum()
        .reset_index()
    )
    summary['recruitment_probability'] = summary['recruited_trials'] / summary['n_trials']
    summary['mean_spikes'] = summary['spike_sum'] / summary['n_trials']
    summary.insert(0, 'mode', mode)
    audit = {
        'mode': mode, 'rows': total_rows,
        'target_axons': summary['target_axon_id'].nunique(),
        'CVs': summary['cv'].nunique(), 'frequencies': summary['freq'].nunique(),
        'regimes': summary['thresh_type'].nunique(),
        'target_condition_groups': len(summary[['target_axon_id', 'cv', 'freq']].drop_duplicates()),
        'missing_amplitudes': missing_amplitudes,
        'missing_spike_counts': missing_spike_counts,
    }
    return summary, audit

true_summary, true_audit = summarize_population_csv(POPULATION_RESPONSES_CSV, 'true')
cv0_summary, cv0_audit = summarize_population_csv(POPULATION_CV0_RESPONSES_CSV, 'cv0')
response_summary = pd.concat([true_summary, cv0_summary], ignore_index=True)
response_audit = pd.DataFrame([true_audit, cv0_audit])
display(response_audit)
display(response_summary.head())

## 4. Population recruitment outcome tables

These tables distinguish two complementary meanings of recruitment. `expected_active_axons` is the mean number active in one repetition (the sum of per-axon recruitment probabilities), whereas `axons_ever_recruited` counts axons that responded in at least one of the 100 repetitions. The former describes a typical population volley; the latter is useful for auditing whether an axon can safely be excluded.

The exported axon-scope and stratum tables compare CV0 replay with the true-amplitude branch. A zero-response boundary in CV0 replay is not automatically a safe pruning boundary for the true branch.

In [ ]:
RECRUITMENT_OUTCOMES_CSV = OUTPUT_DIR / 'population_recruitment_outcomes.csv'
AXON_SCOPE_CSV = OUTPUT_DIR / 'population_axon_recruitment_scope.csv'
STRATUM_SCOPE_CSV = OUTPUT_DIR / 'population_recruitment_scope_by_stratum.csv'
INDEX_PHASE_OUTCOMES_CSV = OUTPUT_DIR / 'index_location_phase_recruitment.csv'

outcome_group_columns = ['mode', 'freq', 'cv', 'thresh_type']
outcome_source = response_summary.copy()
outcome_source['always_recruited'] = outcome_source['recruited_trials'].eq(outcome_source['n_trials'])
recruitment_outcomes = (
    outcome_source.groupby(outcome_group_columns, observed=True, sort=False)
    .agg(
        simulated_axons=('target_axon_id', 'nunique'),
        axons_ever_recruited=('recruited_trials', lambda values: int(values.gt(0).sum())),
        axons_always_recruited=('always_recruited', 'sum'),
        expected_active_axons=('recruitment_probability', 'sum'),
        expected_total_spikes=('mean_spikes', 'sum'),
    )
    .reset_index()
)
recruitment_outcomes['expected_active_fraction'] = (
    recruitment_outcomes['expected_active_axons'] / recruitment_outcomes['simulated_axons']
)
recruitment_outcomes.to_csv(RECRUITMENT_OUTCOMES_CSV, index=False)

axon_mode_scope = (
    response_summary.groupby(
        ['mode', 'target_axon_id', 'target_cutline_id', 'target_diameter_um'],
        observed=True, sort=False,
    )
    .agg(
        ever_recruited=('recruited_trials', lambda values: bool(values.gt(0).any())),
        max_recruitment_probability=('recruitment_probability', 'max'),
        active_condition_cells=('recruited_trials', lambda values: int(values.gt(0).sum())),
    )
    .reset_index()
)
axon_scope = axon_mode_scope.pivot(
    index=['target_axon_id', 'target_cutline_id', 'target_diameter_um'],
    columns='mode',
    values=['ever_recruited', 'max_recruitment_probability', 'active_condition_cells'],
)
axon_scope.columns = [f'{metric}_{mode}' for metric, mode in axon_scope.columns]
axon_scope = axon_scope.reset_index()
for mode in ('cv0', 'true'):
    axon_scope[f'ever_recruited_{mode}'] = axon_scope[f'ever_recruited_{mode}'].astype(bool)
axon_scope['depth_row'] = axon_scope['target_cutline_id'].str[0]
axon_scope['mediolateral_column'] = axon_scope['target_cutline_id'].str[1:].astype(int)
axon_scope['phase_id'] = axon_scope['target_axon_id'].str.extract(r'_(ph\d+)$', expand=False)
axon_scope['within_observed_cv0_response_envelope'] = (
    axon_scope['target_diameter_um'].ge(10.0)
    & (
        (axon_scope['depth_row'].eq('S') & axon_scope['mediolateral_column'].le(9))
        | (axon_scope['depth_row'].eq('M') & axon_scope['mediolateral_column'].le(7))
    )
)
axon_scope['never_recruited_either_branch'] = ~(
    axon_scope['ever_recruited_cv0'] | axon_scope['ever_recruited_true']
)
axon_scope.to_csv(AXON_SCOPE_CSV, index=False)

def summarize_scope(frame, group_column, group_type):
    result = (
        frame.groupby(group_column, observed=True, sort=True)
        .agg(
            n_axons=('target_axon_id', 'size'),
            ever_recruited_cv0=('ever_recruited_cv0', 'sum'),
            ever_recruited_true=('ever_recruited_true', 'sum'),
            never_recruited_either_branch=('never_recruited_either_branch', 'sum'),
        )
        .reset_index()
        .rename(columns={group_column: 'stratum'})
    )
    result.insert(0, 'stratum_type', group_type)
    return result

stratum_scope = pd.concat([
    summarize_scope(axon_scope, 'target_diameter_um', 'diameter_um'),
    summarize_scope(axon_scope, 'depth_row', 'depth_row'),
    summarize_scope(axon_scope, 'mediolateral_column', 'mediolateral_column'),
    summarize_scope(axon_scope, 'target_cutline_id', 'cutline_id'),
], ignore_index=True)
stratum_scope.to_csv(STRATUM_SCOPE_CSV, index=False)

I_ALL_ENDPOINTS = recruitment_outcomes[
    recruitment_outcomes['thresh_type'].eq('I_all')
    & recruitment_outcomes['cv'].isin([0.0, 1.0])
][[
    'mode', 'freq', 'cv', 'simulated_axons', 'axons_ever_recruited',
    'axons_always_recruited', 'expected_active_axons',
    'expected_active_fraction', 'expected_total_spikes',
]].sort_values(['mode', 'freq', 'cv'])

branch_scope = pd.DataFrame({
    'branch': ['cv0', 'true', 'either'],
    'axons_ever_recruited': [
        int(axon_scope['ever_recruited_cv0'].sum()),
        int(axon_scope['ever_recruited_true'].sum()),
        int((axon_scope['ever_recruited_cv0'] | axon_scope['ever_recruited_true']).sum()),
    ],
    'axons_never_recruited': [
        int((~axon_scope['ever_recruited_cv0']).sum()),
        int((~axon_scope['ever_recruited_true']).sum()),
        int(axon_scope['never_recruited_either_branch'].sum()),
    ],
})

index_location_ids = axon_scope.loc[
    axon_scope['target_cutline_id'].eq(index_axon.loc[0, 'cutline_id'])
    & axon_scope['target_diameter_um'].eq(index_axon.loc[0, 'diameter_um']),
    'target_axon_id',
]
index_phase_outcomes = response_summary[
    response_summary['target_axon_id'].isin(index_location_ids)
][[
    'mode', 'target_axon_id', 'cv', 'freq', 'thresh_type',
    'n_trials', 'recruited_trials', 'recruitment_probability', 'mean_spikes',
]].copy()
index_phase_outcomes['phase_id'] = index_phase_outcomes['target_axon_id'].str.extract(
    r'_(ph\d+)$', expand=False
)
index_phase_outcomes.to_csv(INDEX_PHASE_OUTCOMES_CSV, index=False)
index_pair_i_all_10hz = index_phase_outcomes[
    index_phase_outcomes['freq'].eq(10.0)
    & index_phase_outcomes['thresh_type'].eq('I_all')
][[
    'mode', 'cv', 'phase_id', 'recruitment_probability', 'mean_spikes',
]].sort_values(['mode', 'cv', 'phase_id'])
cv0_i_all_10hz_uA = float(index_cv0.loc[index_cv0['freq'].eq(10.0), 'I_all0_uA'].iloc[0])
index_phase_amplitude_margin = index_phase_pair.copy()
index_phase_amplitude_margin['CV0_10Hz_I_all_uA'] = cv0_i_all_10hz_uA
index_phase_amplitude_margin['ph0_reference_minus_CV0_I_all_uA'] = (
    index_phase_amplitude_margin['ph0'] - cv0_i_all_10hz_uA
)

cv0_cutline_diameter = (
    axon_scope[axon_scope['ever_recruited_cv0']]
    .pivot_table(
        index='target_cutline_id', columns='target_diameter_um',
        values='target_axon_id', aggfunc='count', fill_value=0,
    )
)
cv0_envelope = axon_scope['within_observed_cv0_response_envelope']
scope_reduction = pd.DataFrame({
    'scope': ['full population', 'observed CV0 response envelope', 'exact CV0 responder set'],
    'axons_retained': [len(axon_scope), int(cv0_envelope.sum()), int(axon_scope['ever_recruited_cv0'].sum())],
    'CV0 responders_retained': [
        int(axon_scope['ever_recruited_cv0'].sum()),
        int((cv0_envelope & axon_scope['ever_recruited_cv0']).sum()),
        int(axon_scope['ever_recruited_cv0'].sum()),
    ],
    'true responders_retained': [
        int(axon_scope['ever_recruited_true'].sum()),
        int((cv0_envelope & axon_scope['ever_recruited_true']).sum()),
        int((axon_scope['ever_recruited_cv0'] & axon_scope['ever_recruited_true']).sum()),
    ],
})

display(I_ALL_ENDPOINTS)
display(branch_scope)
display(stratum_scope[stratum_scope['stratum_type'].isin(['diameter_um', 'depth_row'])])
display(scope_reduction)
display(cv0_cutline_diameter)
display(index_phase_amplitude_margin)
display(index_pair_i_all_10hz)
print('Wrote:')
print(f'  {RECRUITMENT_OUTCOMES_CSV}')
print(f'  {AXON_SCOPE_CSV}')
print(f'  {STRATUM_SCOPE_CSV}')
print(f'  {INDEX_PHASE_OUTCOMES_CSV}')

## 5. Anatomically registered recruitment maps

Select a response definition, frequency, index regime, and temporal CV below. Each panel uses the same layout as the reference-threshold map: superficial fibers at the top, middle fibers in the center, deep fibers at the bottom, diameters as separate rows, and columns 01–13 from lateral to medial. Within each diameter/location square, ph0 is the upper half and ph1 is the lower half; each phase has its own y-axis tick. Gray cells are anatomically unavailable D01–D04 trajectories rather than nonresponding axons.

In [ ]:
VIEW_MODE = 'cv0'       # 'true' or 'cv0'
VIEW_FREQ_HZ = 100
VIEW_REGIME = 'I_50'
VIEW_CV = 1.0

response_summary['target_phase_id'] = response_summary['target_axon_id'].str.extract(
    r'_(ph\d+)$', expand=False
)
selected = response_summary[
    response_summary['mode'].eq(VIEW_MODE)
    & response_summary['freq'].eq(VIEW_FREQ_HZ)
    & response_summary['thresh_type'].eq(VIEW_REGIME)
    & response_summary['cv'].eq(VIEW_CV)
]
if selected.empty:
    raise ValueError('The selected mode/frequency/regime/CV combination is unavailable.')

recruitment_fig, recruitment_axis, recruitment_map = plot_anatomical_phase_stack(
    selected,
    'recruitment_probability',
    cutline_col='target_cutline_id',
    diameter_col='target_diameter_um',
    phase_col='target_phase_id',
    title=(
        f'{VIEW_MODE.upper()} recruitment probability: {VIEW_REGIME}, '
        f'{VIEW_FREQ_HZ} Hz, CV={VIEW_CV:.1f}'
    ),
    colorbar_label='P(at least one terminal-node spike)',
    cmap='magma',
    vmin=0,
    vmax=1,
    formatter=lambda value: '·' if value == 0 else f'{value:.2f}',
    highlight=index_highlight,
)
plt.show()

## 6. True-regime versus CV0 replay

Summing the per-axon recruitment probabilities gives the expected number of recruited axons in a trial. This comparison keeps the frequency and regime fixed while varying temporal CV.

In [ ]:
COMPARE_FREQ_HZ = 200
COMPARE_REGIME = 'I_all'
population_size = response_summary['target_axon_id'].nunique()
comparison = (
    response_summary[
        response_summary['freq'].eq(COMPARE_FREQ_HZ)
        & response_summary['thresh_type'].eq(COMPARE_REGIME)
    ]
    .groupby(['mode', 'cv'], observed=True)['recruitment_probability']
    .sum()
    .rename('expected_recruited_axons')
    .reset_index()
)
fig, ax = plt.subplots(figsize=(8, 4.5))
for mode, values in comparison.groupby('mode', sort=False):
    ax.plot(values['cv'], values['expected_recruited_axons'], marker='o', label=mode)
ax.set(
    xlabel='Coefficient of variation', ylabel=f'Expected recruited axons (of {population_size})',
    title=f'{COMPARE_REGIME} population recruitment at {COMPARE_FREQ_HZ} Hz',
)
ax.set_ylim(0, population_size)
ax.grid(alpha=0.25)
ax.legend(title='Amplitude definition')
fig.tight_layout()
plt.show()

## 7. Detection behavioral matching with population operating-state mixtures

The population pipeline retains the index axon's nine interpretable amplitude states, but each state now carries the response of the full 420-axon population. Detection thresholds are modeled by mixing the **absolute index-state amplitudes first** and normalizing the resulting curve within frequency, matching the original detection analysis. All mixture weights are non-negative and sum to one.

To match the finalized manuscript analysis, the **quadratic trend through the measured behavioral means is the primary target**. Five flexibility levels are reported: one best single state shared globally; one independently selected but unmixed state per frequency; one global mixture shared across every frequency and CV; one mixture per frequency but shared across CV; and one mixture per frequency/CV point. The per-frequency best-single model separates the benefit of selecting different states across frequencies from the benefit of mixing states within a frequency. The per-CV mixture is a representational upper bound showing whether the simulated operating-state family spans the behavioral trend, not an inferential model of biological mixture probabilities. Direct fitting of the six measurements is retained only as a sensitivity analysis.

The direct-point residual objective includes only the six measured CV values (0, 0.2, ..., 1.0). Predictions are still exported over all 11 simulated CVs for continuity. As in the finalized single-axon analysis, normalization uses the maximum simulated threshold over that complete grid; the normalization audit below quantifies the small indirect scaling contribution from unmeasured intermediate CVs.

The optional observer control uses the quadratic-derived **global** mixture, because it is the closest detection analogue to a stable block-shared state distribution. The frequency-specific one-hot selections are also exported as a parsimonious diagnostic and possible frozen sensitivity control. The upper-bound per-CV weights are never passed to the main discrimination observer. Constrained mixture traces may visually overlap fixed-state models; their incremental fit is therefore reported numerically rather than presented as evidence that detection behavior requires a mixture.

In [ ]:
# This cell performs only lightweight numerical optimization; it does not run NEURON.
import sys

SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from detection.detection_core import run_population_detection_behavioral_matching

MATCHING_DIR = REPO_ROOT / 'results' / 'detection' / 'behavioral_matching'
matching_paths = run_population_detection_behavioral_matching(
    root=REPO_ROOT,
    output_dir=MATCHING_DIR,
    target_kinds=('datapoints', 'quadratic'),
    n_starts=4,
)
behavior_points = pd.read_csv(matching_paths['behavior_points'])
detection_curves = pd.read_csv(matching_paths['curves'])
detection_metrics = pd.read_csv(matching_paths['metrics'])
detection_weights = pd.read_csv(matching_paths['weights'])
frozen_global_weights = pd.read_csv(matching_paths['frozen_global_weights'])
frozen_global_weights_quadratic = pd.read_csv(matching_paths['frozen_global_weights_quadratic'])
frozen_global_weights_datapoints = pd.read_csv(matching_paths['frozen_global_weights_datapoints'])
frozen_per_frequency_best_single = pd.read_csv(
    matching_paths['frozen_per_frequency_best_single']
)
mixture_population_summary = pd.read_csv(matching_paths['mixture_population_summary'])

overall_metrics = detection_metrics[detection_metrics['freq'].astype(str).eq('ALL')][[
    'target_kind', 'mode', 'rmse', 'mae', 'nominal_parameters',
    'gaussian_aic', 'gaussian_bic', 'optimization_success', 'selected_single_regime',
]].sort_values(['target_kind', 'nominal_parameters'])
frequency_metrics = detection_metrics[~detection_metrics['freq'].astype(str).eq('ALL')].copy()
frequency_metrics['freq'] = pd.to_numeric(frequency_metrics['freq'])
quadratic_decision = (
    frequency_metrics[frequency_metrics['target_kind'].eq('quadratic')]
    .groupby('mode', as_index=False)
    .agg(mean_frequency_rmse=('rmse', 'mean'), worst_frequency_rmse=('rmse', 'max'))
)
quadratic_decision = quadratic_decision.merge(
    overall_metrics[overall_metrics['target_kind'].eq('quadratic')][
        ['mode', 'nominal_parameters', 'gaussian_aic', 'gaussian_bic', 'selected_single_regime']
    ], on='mode', how='left', validate='one_to_one',
)
single_rmse = float(quadratic_decision.loc[quadratic_decision['mode'].eq('best_single'), 'mean_frequency_rmse'].iloc[0])
quadratic_decision['RMSE_reduction_vs_single_pct'] = 100.0 * (
    single_rmse - quadratic_decision['mean_frequency_rmse']
) / single_rmse
quadratic_decision['role'] = quadratic_decision['mode'].map({
    'best_single': 'fixed-state benchmark',
    'global': 'frozen observer control',
    'per_frequency': 'constrained detection comparison',
    'per_point': 'representational upper bound only',
})
quadratic_decision = quadratic_decision.sort_values('nominal_parameters')

display(quadratic_decision)
display(frozen_global_weights_quadratic)
display(frozen_per_frequency_best_single)
print('Default frozen control (quadratic global mixture):', matching_paths['frozen_global_weights'])
print('Per-frequency best-single frozen sensitivity:', matching_paths['frozen_per_frequency_best_single'])
print('Direct-point sensitivity weights:', matching_paths['frozen_global_weights_datapoints'])

In [ ]:
# Manuscript-facing comparison: smooth quadratic behavioral trends.
FIT_TARGET_KIND = 'quadratic'
MODE_LABELS = {
    'best_single': 'Best single state',
    'best_single_per_frequency': 'Per-frequency best single state',
    'global': 'Global mixture (frozen control)',
    'per_frequency': 'Per-frequency constrained mixture',
    'per_point': 'Per-CV representational upper bound',
}
MODE_COLORS = {
    'best_single': '#009E73', 'best_single_per_frequency': '#D55E00',
    'global': '#6f2dbd',
    'per_frequency': '#0072B2', 'per_point': '#E69F00',
}
MODE_STYLES = {
    'best_single': ('-', 1.5), 'best_single_per_frequency': ('--', 2.0),
    'global': (':', 1.5),
    'per_frequency': ('-.', 1.7), 'per_point': ('-', 2.5),
}
PRIMARY_PLOT_MODES = ('best_single', 'best_single_per_frequency', 'per_point')
frequencies = sorted(detection_curves['freq'].unique())
fig, axes = plt.subplots(1, len(frequencies), figsize=(20, 4.5), sharex=True, sharey=True)
for axis, frequency_hz in zip(axes, frequencies):
    panel = detection_curves[
        detection_curves['target_kind'].eq(FIT_TARGET_KIND)
        & detection_curves['freq'].eq(frequency_hz)
    ]
    observed = panel[panel['mode'].eq('global') & panel['behavior_mean'].notna()].drop_duplicates('cv')
    target = panel[panel['mode'].eq('global') & panel['behavior_target'].notna()].drop_duplicates('cv').sort_values('cv')
    axis.errorbar(
        observed['cv'], observed['behavior_mean'], yerr=observed['behavior_sem'],
        fmt='o', color='black', capsize=3, label='Behavior mean +/- SEM', zorder=5,
    )
    axis.plot(
        target['cv'], target['behavior_target'], color='black', linestyle='--',
        linewidth=1.7, label='Behavioral quadratic target', zorder=4,
    )
    for mode in PRIMARY_PLOT_MODES:
        values = panel[panel['mode'].eq(mode)].sort_values('cv')
        linestyle, linewidth = MODE_STYLES[mode]
        axis.plot(
            values['cv'], values['prediction'], color=MODE_COLORS[mode],
            linestyle=linestyle, linewidth=linewidth, label=MODE_LABELS[mode],
        )
    axis.set(title=f'{frequency_hz:g} Hz', xlabel='Coefficient of variation', ylim=(0.80, 1.01))
    axis.grid(alpha=0.25)
axes[0].set_ylabel('Normalized detection threshold')
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5, frameon=False)
fig.suptitle('Population detection: global and frequency-specific fixed states versus upper bound', y=1.02)
fig.tight_layout(rect=(0, 0.19, 1, 1))
plt.show()

# Sensitivity analysis: the same models evaluated only at the six measured CV means.
measured_cvs = np.asarray(sorted(behavior_points['cv'].unique()), dtype=float)
fig, axes = plt.subplots(1, len(frequencies), figsize=(20, 4.2), sharex=True, sharey=True)
for axis, frequency_hz in zip(axes, frequencies):
    panel = detection_curves[
        detection_curves['target_kind'].eq('datapoints')
        & detection_curves['freq'].eq(frequency_hz)
    ]
    observed = panel[panel['mode'].eq('global') & panel['behavior_mean'].notna()].drop_duplicates('cv')
    axis.errorbar(
        observed['cv'], observed['behavior_mean'], yerr=observed['behavior_sem'],
        fmt='o', color='black', capsize=3, label='Behavior mean +/- SEM', zorder=5,
    )
    for mode in MODE_LABELS:
        values = panel[panel['mode'].eq(mode) & panel['cv'].isin(measured_cvs)].sort_values('cv')
        linestyle, linewidth = MODE_STYLES[mode]
        axis.plot(
            values['cv'], values['prediction'], color=MODE_COLORS[mode],
            linestyle=linestyle, linewidth=linewidth, marker='o', markersize=3,
            label=MODE_LABELS[mode],
        )
    axis.set(title=f'{frequency_hz:g} Hz', xlabel='Coefficient of variation', ylim=(0.80, 1.01))
    axis.grid(alpha=0.25)
axes[0].set_ylabel('Normalized detection threshold')
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, frameon=False)
fig.suptitle('Direct measured-point sensitivity analysis (not the primary fit target)', y=1.02)
fig.tight_layout(rect=(0, 0.19, 1, 1))
plt.show()

# Normalization audit. The behavioral loss above uses six measured CVs, while the
# denominator follows the finalized single-axon convention: maximum simulated
# threshold over the complete 11-CV grid. Intermediate CVs therefore affect only
# scale, not the set of behavioral residuals.
normalization_rows = []
for (mode, frequency_hz), values in (
    detection_curves[detection_curves['target_kind'].eq('datapoints')]
    .groupby(['mode', 'freq'], sort=True)
):
    full_max = float(values['prediction_absolute_uA'].max())
    measured_max = float(
        values[values['cv'].isin(measured_cvs)]['prediction_absolute_uA'].max()
    )
    max_cv = float(values.loc[values['prediction_absolute_uA'].idxmax(), 'cv'])
    normalization_rows.append({
        'mode': mode, 'freq': frequency_hz, 'full_grid_max_cv': max_cv,
        'maximum_is_measured_cv': bool(np.any(np.isclose(measured_cvs, max_cv))),
        'full_vs_measured_max_percent': 100.0 * (full_max / measured_max - 1.0),
    })
normalization_audit = pd.DataFrame(normalization_rows)
display(normalization_audit.round({'full_grid_max_cv': 1, 'full_vs_measured_max_percent': 3}))

# Absolute fit-error diagnostic: constrained improvements are numerically real but visually small.
quadratic_rmse = (
    frequency_metrics[frequency_metrics['target_kind'].eq('quadratic')]
    .pivot(index='freq', columns='mode', values='rmse')
    .reindex(columns=list(MODE_LABELS))
)
axis = quadratic_rmse.rename(columns=MODE_LABELS).plot(
    kind='bar', figsize=(10, 4.5), color=[MODE_COLORS[mode] for mode in MODE_LABELS],
)
axis.set(
    title='Quadratic-target RMSE by frequency and mixture flexibility',
    xlabel='Frequency (Hz)', ylabel='RMSE',
)
axis.tick_params(axis='x', rotation=0)
axis.grid(axis='y', alpha=0.25)
axis.legend(frameon=False, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# Quadratic-target weight diagnostics. The per-CV stacks expose flexibility; they are not observer weights.
quadratic_weights = detection_weights[
    detection_weights['target_kind'].eq('quadratic')
].copy()
regime_colors = dict(zip(REGIME_ORDER, plt.get_cmap('viridis')(np.linspace(0.05, 0.95, len(REGIME_ORDER)))))

fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
frequency_single_weights = (
    quadratic_weights[quadratic_weights['mode'].eq('best_single_per_frequency')]
    .pivot(index='thresh_type', columns='freq', values='weight')
    .reindex(REGIME_ORDER)
)
single_image = axes[0].imshow(
    frequency_single_weights.to_numpy(), aspect='auto', vmin=0, vmax=1, cmap='Greens'
)
axes[0].set(
    title='Per-frequency best single states', xlabel='Frequency (Hz)', ylabel='Index regime',
    xticks=np.arange(len(frequency_single_weights.columns)),
    xticklabels=[f'{value:g}' for value in frequency_single_weights.columns],
    yticks=np.arange(len(REGIME_ORDER)), yticklabels=REGIME_ORDER,
)
global_values = (
    quadratic_weights[quadratic_weights['mode'].eq('global')]
    .set_index('thresh_type')['weight'].reindex(REGIME_ORDER)
)
axes[1].bar(REGIME_ORDER, global_values, color=[regime_colors[name] for name in REGIME_ORDER])
axes[1].set(title='Quadratic global mixture (frozen control)', ylabel='Weight', ylim=(0, 1))
axes[1].tick_params(axis='x', rotation=45)
frequency_weights = (
    quadratic_weights[quadratic_weights['mode'].eq('per_frequency')]
    .pivot(index='thresh_type', columns='freq', values='weight')
    .reindex(REGIME_ORDER)
)
image = axes[2].imshow(frequency_weights.to_numpy(), aspect='auto', vmin=0, vmax=1, cmap='magma')
axes[2].set(
    title='Quadratic per-frequency constrained mixtures', xlabel='Frequency (Hz)', ylabel='Index regime',
    xticks=np.arange(len(frequency_weights.columns)), xticklabels=[f'{value:g}' for value in frequency_weights.columns],
    yticks=np.arange(len(REGIME_ORDER)), yticklabels=REGIME_ORDER,
)
fig.colorbar(image, ax=axes[2], label='Weight')
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(frequencies), figsize=(20, 4), sharex=True, sharey=True)
for axis, frequency_hz in zip(axes, frequencies):
    values = (
        quadratic_weights[
            quadratic_weights['mode'].eq('per_point') & quadratic_weights['freq'].eq(frequency_hz)
        ]
        .pivot(index='cv', columns='thresh_type', values='weight')
        .reindex(columns=REGIME_ORDER)
        .sort_index()
    )
    axis.stackplot(
        values.index, *[values[name] for name in REGIME_ORDER],
        colors=[regime_colors[name] for name in REGIME_ORDER], labels=REGIME_ORDER,
    )
    axis.set(title=f'{frequency_hz:g} Hz', xlabel='CV', ylim=(0, 1))
axes[0].set_ylabel('Per-CV upper-bound weight')
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=len(REGIME_ORDER), frameon=False)
fig.suptitle('Quadratic per-CV representational upper bound: diagnostic only', y=1.02)
fig.tight_layout(rect=(0, 0.13, 1, 1))
plt.show()

In [ ]:
# Population interpretation of the quadratic global mixture at the behavioral CVs.
behavioral_cvs = sorted(behavior_points['cv'].unique())
global_population_states = mixture_population_summary[
    mixture_population_summary['target_kind'].eq('quadratic')
    & mixture_population_summary['mode'].eq('global')
    & mixture_population_summary['cv'].isin(behavioral_cvs)
][[
    'freq', 'cv', 'mixture_index_amp_uA_mean', 'mixture_expected_active_axons',
    'mixture_expected_active_fraction', 'mixture_expected_total_spikes',
]].sort_values(['freq', 'cv'])
display(global_population_states)

decision_lookup = quadratic_decision.set_index('mode')
global_gain = decision_lookup.loc['global', 'RMSE_reduction_vs_single_pct']
frequency_gain = decision_lookup.loc['per_frequency', 'RMSE_reduction_vs_single_pct']
upper_bound_gain = decision_lookup.loc['per_point', 'RMSE_reduction_vs_single_pct']
print(f'Quadratic global-mixture mean-frequency RMSE reduction versus best single: {global_gain:.1f}%')
print(f'Quadratic per-frequency mean-frequency RMSE reduction versus best single: {frequency_gain:.1f}%')
print(f'Quadratic per-CV upper-bound mean-frequency RMSE reduction versus best single: {upper_bound_gain:.1f}%')
print('Observer handoff: learn block-shared mixtures in Notebook 05; use the quadratic global mixture only as an optional frozen control.')
print('The per-CV mixture remains a detection-only representational upper bound and is never passed to the observer.')
print('All behavioral-matching outputs:')
for name, path in matching_paths.items():
    print(f'  {name}: {path}')

### Interpretation and observer handoff

The quadratic analysis is the manuscript-facing comparison because it tests the smooth behavioral trend used in the finalized single-axon analysis. The best single state asks whether one operating state is sufficient. The global and per-frequency mixtures add constrained levels of heterogeneity. The per-CV mixture is retained only as a representational upper bound: it demonstrates whether the available population states span the behavioral curve, but its CV-specific weights are too flexible to support a biological claim.

The discrimination observer should therefore learn its own block-shared regime mixture. The quadratic global mixture is exported as an optional detection-frozen control, while the per-frequency and per-CV detection weights are not transferred. The direct measured-point fit is a sensitivity analysis and does not replace the quadratic primary target.

## Recomputation

The main NEURON recomputation remains intentionally absent from this review notebook. The completed Longleaf CSVs remain the authoritative production inputs until the operating-state definition is finalized.

The isolated index-axon resolution pilot can be submitted with `slurm/detection/run_index_operating_state_pilot.sh`. It writes only under `results/detection/index_operating_state_pilot/` and does not overwrite production thresholds or responses. Section 7 reruns only the inexpensive constrained behavioral-mixture fits. Once the pilot has resolved numerical tolerance, skipped targets, and nonmonotonicity, use `slurm/detection/run_detection_pipeline.sh` for the intentionally clean full rerun.
